# Modelos de ensemble — custo, biblioteca escolhida e espaços de busca

**Cards 04A (#185) e 04B (#186).** As seções 1 a 5 são do #185 e a seção 6 é do #186, que chegou pela `develop` no mesmo `ensembles.ipynb`.

**Card 04A, issue #185.** Este notebook responde a uma pergunta só: o projeto deve usar o `HistGradientBoostingClassifier`, que já vem no scikit-learn, ou o `XGBClassifier`, que acrescenta uma dependência ao projeto e à sessão do Colab?

A pergunta não é de gosto. Os cards #186, #187 e #188 rodam uma busca aleatória de 40 iterações sobre os folds do contrato, e uma sessão do Colab não dura para sempre. Se um ajuste único custar tempo demais, a busca não cabe na sessão e os três cards seguintes travam. O que decide, então, é medição: quanto custa **um** ajuste completo de cada biblioteca sobre a matriz que o contrato entrega.

**O que este notebook não faz.** Ele não treina o modelo final, não roda busca de hiperparâmetros e não cria nenhuma partição ou validação cruzada própria. As partições são as do contrato do card 01 (`matriz.preparar_matriz`), e nenhuma outra: uma fatia aleatória criada aqui contrariaria o agrupamento por Cliente da seção 3 e ainda mediria um treino menor do que o real, devolvendo um tempo otimista.

## 1. Ambiente e versões

As três bibliotecas que entram na comparação estão fixadas em versão exata no `requirements.txt` desde o commit 1 deste card. O motivo é a própria medição: comparar tempo entre duas bibliotecas só tem sentido se a sessão do Colab e a máquina de quem revisa instalarem a mesma pilha. Com faixa (`>=`), duas execuções do mesmo notebook em dias diferentes podem medir versões diferentes e a diferença de tempo passaria a ser da versão, não da biblioteca.

A célula abaixo instala o `requirements.txt` e imprime as versões efetivamente carregadas.

In [ ]:
!pip install -q -r requirements.txt

import importlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
import xgboost

# Inclui as pastas de código tanto no projeto local quanto no Colab. O break
# evita que duas cópias do mesmo módulo fiquem no caminho de importação.
for pasta_codigo in [Path("src"), Path("../src"), Path("/content/src")]:
    if pasta_codigo.exists():
        sys.path.insert(0, str(pasta_codigo.resolve()))
        break
for pasta_scripts in [Path("scripts"), Path("../scripts"), Path("/content/scripts")]:
    if pasta_scripts.exists():
        sys.path.insert(0, str(pasta_scripts.resolve()))
        break

import ensembles
importlib.reload(ensembles)

# Falha aqui, e nao tres secoes depois, se a sessao carregou uma versao
# diferente da fixada no requirements.txt (revisao da !94).
versoes_conferidas = ensembles.conferir_versoes()

print("scikit-learn", sklearn.__version__)
print("scipy       ", scipy.__version__)
print("xgboost     ", xgboost.__version__)
print("python      ", sys.version.split()[0])
print()
print("conferidas contra o requirements.txt:", versoes_conferidas)

O output acima é a evidência do CR01: as três versões impressas são exatamente as declaradas no `requirements.txt` (`scikit-learn==1.9.1`, `scipy==1.18.1`, `xgboost==3.4.1`), e a última linha mostra o que `ensembles.conferir_versoes` validou. A célula não depende de quem lê reparar numa divergência: se a sessão do Colab carregou uma cópia pré-instalada na frente do arquivo, `conferir_versoes` levanta `AssertionError` com o pacote, a versão esperada e a carregada, e o notebook para aqui. Nesse caso, reinicie o ambiente de execução depois do `pip install` e rode de novo.

A versão do scikit-learn passou de 1.9.0 para 1.9.1 ao incorporar a `develop`, que já fixava a 1.9.1 pelo #206; duas versões exatas da mesma biblioteca no projeto quebrariam a reprodutibilidade que as duas issues pedem.

## 2. Contrato de entrada (card 01)

Tudo o que a medição consome vem de `matriz.preparar_matriz`: as features (`x`), o alvo (`y`), o agrupamento por Cliente (`grupos`), as matrizes já transformadas e o `ColumnTransformer` ajustado **somente no treino**. Nada é reprocessado aqui.

Isso não é formalidade. O pré-processador do contrato imputa com mediana e indicador de ausência, escalona de forma robusta e codifica as categóricas em one-hot; um pré-processamento improvisado neste notebook produziria menos colunas e um ajuste mais rápido do que o que a busca do card #187 vai enfrentar de verdade. O número medido seria otimista justamente na direção que faria a decisão sair errada.

In [ ]:
# No Colab, monte o Drive antes de rodar esta célula:
#   from google.colab import drive; drive.mount('/content/drive')

import matriz
importlib.reload(matriz)
from matriz import preparar_matriz, resumo_da_matriz

CAMINHOS_POSSIVEIS = [
    Path("data/processed/base_analitica.parquet"),
    Path("../data/processed/base_analitica.parquet"),
    Path("/content/data/processed/base_analitica.parquet"),
    Path("/content/drive/MyDrive/data/processed/base_analitica.parquet"),
]
caminho = next((c for c in CAMINHOS_POSSIVEIS if c.exists()), None)
if caminho is None:
    raise FileNotFoundError(
        "Base analítica não encontrada. Rode o pre-processamento.ipynb antes, ou "
        f"coloque o parquet em um destes caminhos: {[str(c) for c in CAMINHOS_POSSIVEIS]}"
    )

df = pd.read_parquet(caminho)

# Datas do registro de decisão da política de particionamento (#127/#128), as
# mesmas do modelagem.ipynb. Elas são parâmetro, nunca constante de módulo.
CORTE_VALIDACAO = "2025-06-01"
CORTE_TESTE = "2025-12-01"

preparo = preparar_matriz(df, CORTE_VALIDACAO, CORTE_TESTE)
matriz_treino = preparo["matrizes"]["treino"]
y_treino = preparo["y"]["treino"]

print(resumo_da_matriz(preparo))
print()
print("colunas da matriz:", preparo["metadados"]["colunas_da_matriz"])
print("Clientes distintos no treino:", preparo["grupos"]["treino"].nunique())

O output acima mostra as três partições com n, intervalo de datas e Clientes. Dois números importam para ler a medição da seção 3: o **n do treino**, porque é sobre ele que o ajuste acontece, e o **número de colunas da matriz**, porque num boosting por histograma o custo depende do número de colunas e de bins muito mais do que do número de linhas.

Repare que o treino é bem menor que a base inteira: o corte é temporal e por Cliente, então o treino é só o período mais antigo. Estimar o custo da busca a partir do total de registros da base superestimaria o tempo; o número que vale é o do treino.

## 3. Medição de custo de um ajuste

`ensembles.medir_bibliotecas` ajusta cada biblioteca **uma** vez sobre a matriz de treino do contrato e devolve tempo de ajuste, pico de memória do processo e a semente usada.

Três cuidados estão embutidos nessa função e são o que torna a comparação honesta:

1. **Hiperparâmetros equivalentes termo a termo.** Mesmo passo, mesmo número de árvores, mesmo número de folhas, mesmo número de bins, histograma nos dois lados. A célula abaixo usa `ensembles.HIPERPARAMETROS_COMPARACAO`, que **não é uma cópia**: passo, número de árvores e número de folhas são lidos de `modelo.HIPERPARAMETROS_CANDIDATO`, os hiperparâmetros do candidato que o projeto realmente treina. A célula imprime os dois lado a lado e falha se divergirem, e o teste `test_comparacao_usa_os_hiperparametros_do_candidato` trava a mesma igualdade fora do notebook.
2. **Sem parada antecipada nos dois lados.** Com ela ligada, o tempo medido seria o de menos árvores do que as declaradas — e, no caso do scikit-learn, a fatia de parada seria aleatória, ignorando o agrupamento por Cliente.
3. **Estimador clonado a cada medição.** Um modelo reaproveitado começaria de um ajuste pronto e pareceria mais barato.

O pico de memória sai do RSS do processo, amostrado em thread separada. Quase toda a alocação das duas bibliotecas é nativa, e `tracemalloc` enxerga só o lado Python: mediria alguns megabytes e diria que as duas custam o mesmo.

In [ ]:
import ensembles
importlib.reload(ensembles)
from ensembles import escolher_biblioteca, estimar_busca, medir_bibliotecas

SEMENTE = ensembles.SEMENTE_PADRAO

import modelo

candidato = modelo.HIPERPARAMETROS_CANDIDATO
print("hiperparâmetros da comparação:", ensembles.HIPERPARAMETROS_COMPARACAO)
print("hiperparâmetros do candidato: ",
      {k: candidato[k] for k in ("learning_rate", "max_iter", "max_leaf_nodes")})

# O tempo medido so e o do modelo treinado se os tres valores forem os mesmos.
assert ensembles.HIPERPARAMETROS_COMPARACAO["passo"] == candidato["learning_rate"]
assert ensembles.HIPERPARAMETROS_COMPARACAO["arvores"] == candidato["max_iter"]
assert ensembles.HIPERPARAMETROS_COMPARACAO["folhas"] == candidato["max_leaf_nodes"]

tabela_medicao = medir_bibliotecas(matriz_treino, y_treino, random_state=SEMENTE)
tabela_medicao[["tempo_ajuste_s", "pico_memoria_mb", "delta_memoria_mb", "random_state"]]

O output acima é a tabela do CR04. Ela tem uma linha por biblioteca, com o tempo de um ajuste completo em segundos, o pico de memória do processo em MB, o quanto desse pico foi acrescentado pelo próprio ajuste (`delta_memoria_mb`) e o `random_state` usado.

O `delta` está ali porque o pico sozinho engana: ele inclui a base carregada e a matriz transformada, que já estavam na memória antes do ajuste e são as mesmas para as duas bibliotecas. Quem diferencia uma da outra é o delta.

### 3.1. Medição registrada

Execução de 17/09/2026, semente 42, scikit-learn 1.9.0 (antes do alinhamento à 1.9.1 descrito na seção 1), sobre uma matriz de treino de 167.815 linhas e 31 colunas:

| Biblioteca | Tempo de um ajuste (s) | Pico de memória (MB) | Acréscimo do ajuste (MB) | `random_state` |
| --- | ---: | ---: | ---: | ---: |
| `HistGradientBoostingClassifier` | 3,58 | 582,4 | 33,7 | 42 |
| `XGBClassifier` | 2,65 | 567,4 | 22,5 | 42 |

**Ressalva sobre esta execução, que precisa ser lida antes de usar os números.** Não havia base da Azul na máquina onde a tabela acima foi medida, e o `.gitignore` do projeto proíbe versioná-la. Os números vieram de uma base **sintética com a mesma forma declarada na issue** — 484.915 registros e 407.139 valores distintos de `ID_GOLDENRECORD`, passando pelo mesmo `preparar_matriz` e produzindo as mesmas 31 colunas de matriz. O que isso mede bem é a ordem de grandeza e a razão entre as duas bibliotecas; o que isso não mede é o efeito da distribuição real das categorias sobre o número de bins efetivos.

Além disso, a medição rodou em máquina local com 14 CPUs lógicas e 15,5 GB de RAM. Uma sessão gratuita do Colab tem cerca de 2 vCPUs, e as duas bibliotecas paralelizam o ajuste: espere algo entre 5 e 7 vezes mais lento lá.

**Portanto:** ao rodar este notebook no Colab sobre a base real, substitua a tabela acima pelo output da célula da seção 3 e refaça a conta da seção 4. A decisão da seção 5 continua válida enquanto a conclusão da seção 4 continuar valendo — e a seção 4 diz explicitamente qual é o limite.

## 4. A busca dos cards seguintes cabe na sessão?

`estimar_busca` multiplica o tempo de um ajuste pelas 40 iterações da busca e pelo número de folds do contrato. A conta é deliberadamente otimista: conta só os ajustes e ignora transformação, cálculo de métrica e o fato de cada fold custar quase o treino inteiro. Serve como piso — se nem esse piso couber na sessão, a busca com certeza não cabe.

O orçamento de 3 horas é o que se assume disponível numa sessão de Colab antes de ela cair ou ser reciclada.

In [ ]:
ORCAMENTO_HORAS = 3.0
N_ITERACOES = 40
N_FOLDS = 5

for biblioteca, linha in tabela_medicao.iterrows():
    horas = estimar_busca(linha["tempo_ajuste_s"], N_ITERACOES, N_FOLDS)
    print(f"{biblioteca:32s} {horas:6.2f} h  ({N_ITERACOES} iterações x {N_FOLDS} folds)")

decisao = escolher_biblioteca(
    tabela_medicao,
    orcamento_horas=ORCAMENTO_HORAS,
    n_iteracoes=N_ITERACOES,
    n_folds=N_FOLDS,
)
print()
print("escolhida:", decisao["escolhida"])
print(decisao["justificativa"])

O output acima aplica o critério do CR05, e a ordem dele importa: primeiro descarta quem não cabe no orçamento da sessão, porque uma biblioteca que estoura a sessão não é uma opção mais lenta, é uma opção que não existe; entre as que cabem, vence a mais barata de manter.

Na execução registrada, as duas cabem com folga — 0,20 h para o `HistGradientBoostingClassifier` e 0,15 h para o `XGBClassifier`, contra um orçamento de 3 h. Mesmo aplicando o fator de 5 a 7 vezes do Colab, as duas seguem abaixo de 1,5 h. **O limite onde essa conclusão mudaria:** um ajuste precisaria custar mais de 54 segundos no ambiente onde a busca for rodar (3 h ÷ 200 ajustes). Se o output da sua execução passar disso, a decisão da seção 5 precisa ser refeita.

## 5. Decisão registrada

**O projeto usa o `HistGradientBoostingClassifier` como biblioteca de boosting dos modelos de ensemble.**

A justificativa tem duas partes, nesta ordem:

1. **Viabilidade.** As duas bibliotecas cabem no orçamento da sessão com margem larga. A busca de 40 iterações vezes 5 folds sai em 0,20 h com o `HistGradientBoostingClassifier` e 0,15 h com o `XGBClassifier` na medição registrada; mesmo no pior fator esperado do Colab, nenhuma das duas se aproxima das 3 h. Ou seja, o custo **não** é critério de desempate aqui — a diferença de 0,93 s por ajuste não muda o que é possível fazer.
2. **Custo de manutenção.** Como a viabilidade não separa as duas, decide o que sobra: o `HistGradientBoostingClassifier` já está no scikit-learn, que o projeto usa desde a seção 4.2.2, e não acrescenta dependência ao `requirements.txt` nem instalação à sessão do Colab. O `XGBClassifier` acrescenta as duas coisas em troca de 26% de tempo num ajuste que já custa segundos.

**O que faria esta decisão ser revista.** O `XGBClassifier` foi medido mais rápido, e a vantagem dele cresce com o tamanho do problema. Se um card futuro aumentar a matriz — mais features, mais categorias, mais bins — a ponto de um ajuste passar dos 54 segundos no ambiente de execução, a ordem do critério se inverte e a dependência extra passa a se pagar. Por isso o `xgboost` fica declarado no `requirements.txt` mesmo não sendo o escolhido agora: a medição precisa continuar reproduzível para quem refizer essa conta.

**Comunicação.** Esta decisão vale para os cards #186, #187 e #188 e precisa chegar à dupla de Métricas e Decisões, que depende dela para dimensionar a busca.

## 6. Espaços de busca da busca aleatória (card 04B, #186)

As seções 1 a 5 mediram **quanto custa** um ajuste e escolheram a biblioteca de boosting. Esta seção declara **em que intervalo** cada hiperparâmetro pode variar na busca dos cards seguintes, e é o conteúdo que veio do #186, com a numeração ajustada para caber neste notebook.

Este notebook documenta e justifica os espaços de busca aleatória (`RandomizedSearchCV`) do Random Forest e do Gradient Boosting, definidos em `src/ensembles.py`. Ele **não executa nenhuma busca**: treinar as combinações nos folds do #102 e comparar contra os pisos fica para os cards que dependem deste — #187 (Random Forest), #188 e #189 (Gradient Boosting) e #190.

**Pontos de partida em comum com o candidato único da seção 4.3.** A base analítica tem 484.915 registros e 20,44% de Detratores (seção 4.2.1 da documentação); o treino usado pela validação cruzada por Cliente do #102, depois do split, tem 341.962 linhas. O contrato de features do card 01 (`FEATURE_SET_V1`, em `scripts/preprocessamento_nps.py`) declara 11 features brutas, e é o mesmo contrato usado aqui.

**Gradient Boosting: duas bibliotecas, por enquanto.** O #185 ainda não decidiu entre `HistGradientBoostingClassifier` (scikit-learn) e `XGBClassifier` (XGBoost) para o trabalho de Ensemble, então este notebook e `src/ensembles.py` mantêm os dois espaços lado a lado. No dia em que o #185 decidir, a biblioteca descartada sai dos dois lugares no mesmo commit.

**Os testes que garantem que os espaços funcionam** ficam em `tests/test_ensembles.py`, e não neste notebook: eles sorteiam 40 combinações de cada espaço com semente fixa e instanciam o estimador correspondente, o que é rápido demais para justificar rodar aqui a cada execução do notebook. Este notebook sorteia só uma amostra pequena, para inspeção visual dos valores.

### 6.1. Random Forest

#### 6.1.1. Espaço de busca declarado

A célula abaixo importa `ESPACO_RANDOM_FOREST` de `src/ensembles.py` e lista os cinco eixos declarados. A importação do módulo não ajusta nenhum estimador (CR01): o que a célula imprime é a distribuição em si, não uma amostra dela.

In [ ]:
# O caminho de importacao ja foi preparado na secao 1.
from ensembles import ESPACO_RANDOM_FOREST


def descrever(distribuicao):
    """Nome e parâmetros da distribuição, em vez do endereço de memória do repr padrão.

    O repr padrão de um objeto `scipy.stats` traz o endereço em memória, que muda
    a cada execução e sujaria o diff do notebook sem nenhuma informação nova.
    """
    if isinstance(distribuicao, list):
        return f"lista com {len(distribuicao)} opções: {distribuicao}"
    return f"scipy.stats.{distribuicao.dist.name}, parâmetros={distribuicao.args or distribuicao.kwds}"


for nome, distribuicao in ESPACO_RANDOM_FOREST.items():
    print(f"{nome}: {descrever(distribuicao)}")

**Leitura da saída.** Cinco eixos: `n_estimators`, `max_depth` e `min_samples_leaf` são inteiros sorteados por `scipy.stats.randint`; `max_features` é uma fração contínua sorteada por `scipy.stats.uniform`; `class_weight` é a única lista simples do espaço, porque reponderação de classe é uma escolha categórica, sem escala, e não uma distribuição numérica. A tabela consolidada da seção 3 traz o intervalo e a justificativa de cada um.

#### 6.1.2. Amostra de combinações, para inspeção visual

`ParameterSampler` sorteia cinco combinações do espaço com semente fixa. Cinco é o suficiente para olhar os valores a olho nu; a garantia de que 40 combinações são aceitas pelo `RandomForestClassifier` é o que `tests/test_ensembles.py` cobre, não esta célula.

In [ ]:
import pandas as pd
from sklearn.model_selection import ParameterSampler

SEMENTE = 42

amostra_rf = pd.DataFrame(
    ParameterSampler(ESPACO_RANDOM_FOREST, n_iter=5, random_state=SEMENTE)
)
amostra_rf

**Leitura da tabela.** Cada linha é uma combinação que um `RandomizedSearchCV` real sortearia e passaria ao `RandomForestClassifier`. `max_features` sai como fração entre 0,3 e 1,0, nunca uma contagem de colunas; `class_weight` alterna entre as três opções declaradas. Os valores concretos mudam se a semente mudar, mas a mesma semente reproduz sempre esta tabela, como `test_amostragem_e_reprodutivel_com_a_mesma_semente` confere.

### 6.2. Gradient Boosting

Duas bibliotecas convivem aqui até o #185 decidir. As duas seções abaixo seguem a mesma estrutura da seção 1: espaço declarado, depois amostra pequena para inspeção.

#### 6.2.1. `HistGradientBoostingClassifier` (scikit-learn)

Mesma biblioteca do candidato único da seção 4.3 do notebook `modelagem.ipynb`. A célula lista `ESPACO_GRADIENT_BOOSTING_HISTGB`.

In [ ]:
from ensembles import ESPACO_GRADIENT_BOOSTING_HISTGB

for nome, distribuicao in ESPACO_GRADIENT_BOOSTING_HISTGB.items():
    print(f"{nome}: {descrever(distribuicao)}")

**Leitura da saída.** Cinco eixos: `learning_rate` é log-uniforme, porque o efeito do passo sobre o número de iterações necessário é multiplicativo; `max_leaf_nodes`, `max_iter` e `min_samples_leaf` são inteiros; `class_weight` é lista, pelo mesmo motivo da seção 1. `early_stopping` não entra no espaço — continua fixo em `False` em `criar_candidato` (`src/modelo.py`), porque ligá-lo vazaria Cliente entre ajuste e medição interna, e isso não é uma escolha que a busca deva reabrir.

#### 6.2.2. `XGBClassifier` (XGBoost)

A célula lista `ESPACO_GRADIENT_BOOSTING_XGBOOST`. Sete eixos, dois a mais que o espaço do HistGB, porque o XGBoost expõe sub-amostragem de linhas e de colunas como parâmetros próprios, que o HistGB não tem por discretizar em histogramas em vez de amostrar.

In [ ]:
from ensembles import ESPACO_GRADIENT_BOOSTING_XGBOOST

for nome, distribuicao in ESPACO_GRADIENT_BOOSTING_XGBOOST.items():
    print(f"{nome}: {descrever(distribuicao)}")

**Leitura da saída.** `learning_rate` usa a mesma distribuição log-uniforme do HistGB, para o custo por ajuste do #185 comparar as duas bibliotecas sob o mesmo raciocínio de passo. `scale_pos_weight` é o único eixo com teto calculado a partir da base: `RAZAO_DESBALANCEAMENTO` (~3,89) é a proporção de não-Detratores por Detrator, e um `scale_pos_weight` nesse valor neutraliza o desbalanceamento por completo.

#### 6.2.3. Amostra combinada, para inspeção visual

Uma amostra pequena de cada biblioteca, lado a lado.

In [ ]:
amostra_histgb = pd.DataFrame(
    ParameterSampler(ESPACO_GRADIENT_BOOSTING_HISTGB, n_iter=5, random_state=SEMENTE)
)
amostra_xgboost = pd.DataFrame(
    ParameterSampler(ESPACO_GRADIENT_BOOSTING_XGBOOST, n_iter=5, random_state=SEMENTE)
)
print("HistGradientBoostingClassifier:")
display(amostra_histgb)
print("XGBClassifier:")
display(amostra_xgboost)

**Leitura das tabelas.** As duas amostram a mesma semente e o mesmo `n_iter`, mas eixos diferentes: `max_leaf_nodes` de um lado, `max_depth` do outro, porque cada biblioteca limita o tamanho da árvore à sua maneira. Nenhuma das duas tabelas depende da outra ter rodado antes — os dois espaços são independentes, e é isso que permite descartar um deles no dia em que o #185 decidir sem tocar no que sobra.

### 6.3. Tabela consolidada de hiperparâmetros (CR04)

| Modelo | Hiperparâmetro | Distribuição | Intervalo | Justificativa |
|---|---|---|---|---|
| Random Forest | `n_estimators` | `randint` | 200 a 600 | Com 341.962 linhas de treino a variância de cada árvore já é pequena; acima de algumas centenas o ganho de agregar mais árvores fica marginal e o custo segue linear no número delas. |
| Random Forest | `max_depth` | `randint` | 3 a 20 | Teto próximo de log₂(341.962) ≈ 18,4: uma árvore balanceada raramente precisa de mais níveis para separar o treino inteiro. Profundidade ilimitada arriscaria árvores memorizando Cliente, o mesmo risco documentado para `min_samples_leaf` no #103. |
| Random Forest | `min_samples_leaf` | `randint` | 1 a 100 | 100 é o valor que o #103 escolheu para o Gradient Boosting nesta mesma base; o piso em 1 mantém a ponta não regularizada disponível para a busca comparar. |
| Random Forest | `max_features` | `uniform` | 0,3 a 1,0 (fração) | O contrato do card 01 declara 11 features brutas; `"sqrt"`/`"log2"` datam de bases com centenas de colunas e amostrariam ~3 delas por split, descartando a maior parte do sinal disponível. |
| Random Forest | `class_weight` | lista | `balanced`, `balanced_subsample`, `None` | 20,44% de detração é desbalanceamento moderado. Ao contrário do candidato único (onde reponderar prejudicaria a probabilidade calibrada do #103), o Random Forest vota por árvore e não expõe a mesma dependência direta da calibração — a busca decide empiricamente qual das três funciona melhor nesta base. |
| Gradient Boosting (HistGB) | `learning_rate` | `loguniform` | 0,01 a 0,3 | O #103 fixou 0,05 e a grade do #104 testou 0,05/0,10; escala log porque o efeito do passo sobre o número de iterações necessário é multiplicativo. |
| Gradient Boosting (HistGB) | `max_leaf_nodes` | `randint` | 15 a 127 | Cobre o padrão da biblioteca (31, usado pelo #103) e o ponto testado pela grade do #104 (63), com margem para os dois lados. |
| Gradient Boosting (HistGB) | `max_iter` | `randint` | 100 a 600 | Cobre os três pontos da grade do #104 (150, 300, 600). Sem parada antecipada, é o único limite do ensemble. |
| Gradient Boosting (HistGB) | `l2_regularization` | `uniform` | 0,0 a 2,0 | O #103 fixou 1,0 contra o padrão 0,0 da biblioteca; o intervalo cobre os dois extremos e o dobro da escolha do #103. |
| Gradient Boosting (HistGB) | `min_samples_leaf` | `randint` | 20 a 200 | Mesmo raciocínio do Random Forest: folha pequena sobre 341.962 linhas memoriza Cliente. O piso de 20 é o padrão da biblioteca. |
| Gradient Boosting (HistGB) | `class_weight` | lista | `None`, `balanced` | O #103 documenta por que `class_weight=None` foi a escolha do candidato: reponderar deslocaria a probabilidade predita, piorando o Brier da seção 6 e o limiar da seção 7. O eixo entra na busca para que a comparação empírica sustente essa escolha, não só o argumento teórico. |
| Gradient Boosting (XGBoost) | `learning_rate` | `loguniform` | 0,01 a 0,3 | Mesma faixa e mesma razão log-uniforme do HistGB, por serem as duas gradient boosting sobre árvores. |
| Gradient Boosting (XGBoost) | `max_depth` | `randint` | 3 a 10 | O XGBoost limita a árvore por profundidade, e não por número de folhas; 10 níveis já produzem até 2¹⁰ folhas, ordem de interação comparável ao teto de 127 folhas do espaço irmão. |
| Gradient Boosting (XGBoost) | `n_estimators` | `randint` | 100 a 600 | Mesmo intervalo de `max_iter` do HistGB, para que o custo por ajuste medido no #185 seja comparável entre as duas bibliotecas sob o mesmo orçamento de iterações. |
| Gradient Boosting (XGBoost) | `min_child_weight` | `randint` | 1 a 20 | Equivalente do XGBoost a `min_samples_leaf`: soma mínima de peso das observações numa folha antes dela poder ser criada, mesma lógica de regularização contra folha memorizando Cliente. |
| Gradient Boosting (XGBoost) | `reg_lambda` | `uniform` | 0,0 a 2,0 | Mesma faixa de `l2_regularization` do HistGB, pelo nome equivalente no XGBoost. |
| Gradient Boosting (XGBoost) | `subsample` | `uniform` | 0,6 a 1,0 | Mecanismo de regularização do XGBoost sem par direto no HistGB (que discretiza em histogramas em vez de amostrar linhas). O piso de 0,6 evita perder informação suficiente para exigir muito mais iterações. |
| Gradient Boosting (XGBoost) | `colsample_bytree` | `uniform` | 0,5 a 1,0 | Equivalente a `max_features` do Random Forest: com 11 features brutas no contrato do card 01, um piso baixo deixaria cada árvore enxergar poucas colunas. |
| Gradient Boosting (XGBoost) | `scale_pos_weight` | `uniform` | 1,0 a ~3,89 | O teto é `RAZAO_DESBALANCEAMENTO`, a proporção observada de não-Detratores por Detrator (385.775 / 99.140). O XGBoost não tem uma versão `"balanced"` pronta como o scikit-learn; este parâmetro multiplica o gradiente da classe positiva, e o teto no valor exato da proporção é o ponto em que a reponderação neutraliza o desbalanceamento por completo. |

**O que fica para os cards seguintes.** Este notebook não mede custo por ajuste nem escolhe `n_iter` para a `RandomizedSearchCV` real — isso depende do #185, que ainda está aberto. Quando ele decidir a biblioteca de Gradient Boosting, a seção 2 deste notebook e o espaço correspondente em `src/ensembles.py` perdem a metade descartada no mesmo commit.

## 7. Dependências e próximos passos

**Incorpora:** o #186 (espaços de busca), que passou a viver na seção 6 deste mesmo notebook quando as duas frentes se encontraram em `src/ensembles.py`.

**Depende de:** card 01 (contrato de dados, Kaylan e Pedro) — `x`, `y`, `grupos` e o `ColumnTransformer`, consumidos na seção 2 via `matriz.preparar_matriz`.

**Bloqueia:** #186, #187 e #188, que só podem dimensionar a busca depois da decisão da seção 5.

**Próximo passo obrigatório:** rodar este notebook em sessão nova do Colab, sobre a base analítica real, e substituir a tabela da seção 3.1 pelo output medido lá. As travas que protegem a validade da medição estão em `tests/test_ensembles.py` e rodam com `pytest tests/test_ensembles.py -v`.